In [13]:
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from nbafuns import *

data_DIR = "../data/salary/"

In [42]:
def get_spotrac(season=2024):
    url = f'https://www.spotrac.com/nba/rankings/player/_/year/{season}/sort/cap_total'
    all_players = []
    header = {
                "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
                "X-Requested-With": "XMLHttpRequest",
            }
    response = requests.get(url,headers=header)
    if response.status_code != 200:
        print(f"WARNING! Status Code = {response.status_code}")
        
    soup = BeautifulSoup(response.content, 'html.parser')

    # Locate the list containing the data
    table_div = soup.select_one('#table > div.col-lg-8.col-md-8.col-sm-12 > ul')

    # Check if the table exists
    if table_div:
        rows = []
        
        # Loop through list items
        for rank, item in enumerate(table_div.find_all('li')):
            text = item.get_text(separator="|", strip=True)  # Separate using "|"
            parts = text.split("|")  # Split into list

            #If Rank is missing, insert it
            if len(parts) == 3:
                parts.insert(0,rank+1)
                
            rows.append(parts)
        
        # Define column names
        columns = ['Rank', 'Player', 'Team, Pos', 'Cap Hit']

        # Create DataFrame
        df = pd.DataFrame(rows, columns=columns)

        # Split 'Team_Pos' column into 'Team' and 'Position'
        df[['Team', 'Position']] = df['Team, Pos'].str.split(', ', expand=True)

        # Drop the original combined column
        df = df.drop(columns=['Team, Pos'])[['Rank','Player','Team','Position','Cap Hit']]

        #Drop Null Rows (Separators on the website)
        df = df.dropna(thresh=3)
        df1 = df.drop(columns="Rank").reset_index(drop=True)
        df1["Salary"]=df1["Cap Hit"].str.replace("$",'')
        df1["Salary"] = df1["Salary"].str.replace(",",'')
        df1["Salary"] = df1["Salary"].astype(int)
        df1 = df1.drop(columns="Cap Hit")
        df1["Season"] = season + 1
        df1.to_parquet(data_DIR + f"NBA_Spotrac_{season}.parquet")
        return df1


In [43]:
for season in tqdm(range(2010,2025)):
    df1 = get_spotrac(season)
    time.sleep(5)

100%|██████████| 15/15 [01:38<00:00,  6.59s/it]
